# Current Advances in LLM Reasoning — live demo notebook

The hands-on companion to the ACL 2026 tutorial ([llmreasoning.github.io](https://llmreasoning.github.io)). Each section is a short, runnable story:

1. **Instability** — do models even reason _consistently_?
2. **Test-time strategies** — spending inference compute to reason _better_.
3. **Post-training** — how SFT / preference / RL stages reshape one model.
4. **Budget forcing** — directly controlling how long the model thinks.
5. **Multilinguality** — does the model reason in the language you ask in?


In [1]:
%load_ext autoreload
%autoreload 2
import asyncio
from demo.reasoning_demo.client import LLM
from demo.reasoning_demo import mgsm as M
from demo.reasoning_demo import traces as T   # loads the shipped s1.1 generations
from IPython.display import HTML, display
import demo.instability.consistency as C

import json
from IPython.display import display

from helpers import show, extract_boxed, think_span

## 1. Instability in LLM reasoning

Before we try to make models reason _better_, we should ask whether they reason _consistently_ at all. Across this section the model, the problem, and the settings stay fixed — only the sampling seed changes — and the answer moves anyway.


### 1.1 What's the instability in LLM reasoning

We ask the **same model** the **same question** with the **same settings**, twice in a row. Nothing changes between the two calls except the sampling randomness.

Insights:

- Same prompt, same decoding parameters. Yet the two runs can land on different answers (or reach the same one by different work).
- This isn't a bug: sampling makes generation stochastic. The question is how much it moves the _answer_, not just the wording.


In [2]:
llm = LLM()

PROBLEM = 2   # which MGSM problem (0–249); the same index is the same problem in every language
prob_en = M.load_mgsm('en', n=PROBLEM + 1)[PROBLEM]


print('REQUEST:')
r_en = await M.asolve_direct(llm, prob_en.question, 'en')
print(r_en.detail['prompt'])
print('\nRESPONSE 1:')
print(r_en.text)
ok = 'correct' if M.is_correct(r_en.answer, prob_en.answer) else 'WRONG'
print(f'\n[extracted: {r_en.answer}  |  gold: {prob_en.answer}  |  {ok}]')

print('\nRESPONSE 2:')
r_en = await M.asolve_direct(llm, prob_en.question, 'en')
print(r_en.text)
ok = 'correct' if M.is_correct(r_en.answer, prob_en.answer) else 'WRONG'
print(f'\n[extracted: {r_en.answer}  |  gold: {prob_en.answer}  |  {ok}]')

REQUEST:
Question: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?
Answer:

RESPONSE 1:
## Step 1: Calculate the original value of the house after repairs.
The original value of the house was $80,000. Josh put in $50,000 in repairs, which increased the value of the house by 150%. This means the value of the house after repairs is 150% of the original value plus the original value.

## Step 2: Calculate 150% of the original value of the house.
150% of $80,000 is 1.5 * $80,000 = $120,000.

## Step 3: Add the original value of the house to the 150% increase to find the new value of the house.
The new value of the house is $80,000 + $120,000 = $200,000.

## Step 4: Calculate the profit made by Josh.
The profit made by Josh is the new value of the house minus the original cost of the house plus the repairs. The original cost of the house was $80,000, and the 

### 1.2 How severe can the instability be?

The interactive Game 24 demo makes it concrete: one model, one puzzle, many identical runs, overlaid on a single tree. Run it live with the command below, or read the snapshot underneath.

Insights:

- The valid runs collapse onto one shared trunk; the failures peel off exactly where the model fabricates a number it was never given.
- Same puzzle, same settings. The spread of outcomes _is_ the instability, and made visual, looks significant.


`uv run python demo/serve.py --open game24 --no-strategies`


![Game 24](assets/game24.png)


### 1.3 How often this happens in practise?

Zooming out from a single puzzle to a whole grid: each model × benchmark cell is 30 reruns × 50 problems, drawn as a solved-or-not (0/1) heatmap.

Insights:

- A mottled rather than solid heatmap means the model flips between right and wrong across seeds on the _same_ problems.
- Instability is specific: a model can be flaky on one benchmark but steady on another, and a benchmark flaky for one model but not another.


In [3]:
MODEL_1 = 'gpt-oss-120b'
MODEL_2 = 'gemini-3-flash'
MODEL_3 = "gpt-4.1-mini"

BENCHMARK_1 = 'game24' 
BENCHMARK_2 = "scibench"

C.show_heatmap(MODEL_1,     BENCHMARK_1)

This doesn't mean that the _same model_ is unstable across all benchmarks


In [4]:
C.show_heatmap(MODEL_1,     BENCHMARK_2)

The also doesn't mean that the _same benchmark_ is unstable for all models


In [5]:
C.show_heatmap(MODEL_2,     BENCHMARK_1)

But it surely happens often.


In [6]:
C.show_heatmap(MODEL_2,     "hle")
C.show_heatmap(MODEL_3,     "game24")
C.show_heatmap(MODEL_1,     "sonnetwriting")

### 1.4 Does instability matter?

If a model is right on some seeds and wrong on others, the single number a benchmark reports is a _choice_ of scoring protocol, not a fact about the model.

Insights:

- The same reruns scored as single-run / mean@N / maj@N / pass@N give very different "capability".
- maj@N (majority vote) and pass@N (any run succeeds) turn that flakiness into higher reported scores — drag N in the demo to watch it move.


`uv run python demo/serve.py --open protocols --no-strategies`


In [7]:
C.show_protocols('game24')

## 2. Comparing budget utilization across reasoning strategies

One way to make a model reason _better_ is simply to spend more compute at inference time. Here we take a single GSM8K problem and solve it with seven test-time strategies, then compare how each one spends its budget.

Insights:

- The budget shows up three ways: **latency** (wall-clock), **tokens**, and **calls** (round-trips to the model).
- There's no single best strategy. The winner depends on the criteria you care most about such as latency, accuracy, ease of implementation, and generalizability.


`uv run python demo/serve.py --open strategies`


In [ ]:
# The notebook version of the interactive demo: pick one GSM8K problem and solve
# it with each test-time strategy, then compare how each spends its budget.
from demo.reasoning_demo.data import load_problems
from demo.reasoning_demo.extract import is_correct
from demo.reasoning_demo.strategies import (
    input_output, self_consistency, react, agentic,
    iterative, tree_of_thoughts, fleet_of_agents,
)

llm = LLM()
problems = load_problems() # bundled GSM8K sample
prob = next((p for p in problems if p.id == 'gsm8k-3'), problems[0])

print(f'Model: {llm.model}')
print(f'Problem [{prob.id}]:\n  {prob.question}')
print(f'Gold answer: {prob.answer}\n')

# same problem, seven ways to spend inference compute (sampling, searching, refining, tools)
methods = [
    ('input-output (direct)',  lambda: input_output(llm, prob.question)),
    ('self-consistency k=5',   lambda: self_consistency(llm, prob.question, k=5)),
    ('react (<=6 steps)',      lambda: react(llm, prob.question, max_steps=6)),
    ('agent (tool-use)',       lambda: agentic(llm, prob.question, max_steps=6)),
    ('self-refine (2 rounds)', lambda: iterative(llm, prob.question, rounds=2)),
    ('tree-of-thoughts d=2',   lambda: tree_of_thoughts(llm, prob.question, depth=2)),
    ('fleet-of-agents n=3',    lambda: fleet_of_agents(llm, prob.question, n_agents=4)),
]

print(f"{'method':<24}{'answer':>8}{'correct':>9}{'latency':>10}{'tokens':>9}{'calls':>7}")
print('-' * 67)
for label, run in methods:
    r = run()
    ok = 'yes' if is_correct(r.answer, prob.answer) else 'NO'
    print(f'{label:<24}{str(r.answer):>8}{ok:>9}'
          f'{r.latency_s:>9.2f}s{r.usage.total_tokens:>9}{r.usage.calls:>7}')

Model: meta-llama/llama-3.1-8b-instruct
Problem [gsm8k-3]:
  James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  How many total meters does he run a week?
Gold answer: 540

method                    answer  correct   latency   tokens  calls
-------------------------------------------------------------------
input-output (direct)        540      yes     1.29s       90      1
self-consistency k=5         540      yes     6.97s     1061      5
react (<=6 steps)            540      yes    11.38s     3511      7
agent (tool-use)             540      yes     4.35s      966      2
self-refine (2 rounds)       540      yes     9.91s     2847      5
tree-of-thoughts d=2         540      yes     6.00s     2854     13
fleet-of-agents n=4          540      yes    10.74s     5198     25


## 3. Comparing the effects of different Post-training methods


We take one base model **InternVL3.5-1B** — and observe four _cumulative_ post-training stages, then ask the **same** question at each stage and watch the answer (and the reasoning behind it) change:

1. **Pretrained** — next-token prediction only, no alignment
2. **Pretrained + SFT** — supervised fine-tuning on instruction → response data
3. **Pretrained + SFT + MPO** — Mixed Preference Optimization (preference alignment)
4. **Pretrained + SFT + MPO + GSPO** — Group Sequence Policy Optimization (a final RL stage)

The task is a **MathVerse "Vision Only"** geometry problem — the question _and_ the answer choices are baked into the figure, so the image below is **all** the model sees.


In [8]:
run = json.load(open('results/post_run.json'))
sample = run['sample']

# map each saved model id -> its post-training stage, and index the runs by stage
STAGE = {
    'OpenGVLab/InternVL3_5-1B-Pretrained': 'Pretrained',
    'OpenGVLab/InternVL3_5-1B-Instruct':   'Pretrained + SFT',
    'OpenGVLab/InternVL3_5-1B-MPO':        'Pretrained + SFT + MPO',
    'OpenGVLab/InternVL3_5-1B':            'Pretrained + SFT + MPO + GSPO',
}
runs = {STAGE[r['model']]: r for r in run['results']}



print(f"{run['dataset']['name']} · {sample['problem_version']} · "
      f"{sample['question_type']} · gold answer = {sample['gold_answer']}")

AI4Math/MathVerse · Vision Only · multi-choice · gold answer = D


![MathVerse](assets/mathverse_sample.png)


### Stage 1 — Pretrained

Only next-token prediction on web-scale data, no alignment. It barely _answers_: it emits a short multiple-choice-looking string and stops.


In [9]:
show(runs, 'Pretrained')

<IPython.core.display.Latex object>

### Stage 2 — + SFT

Supervised fine-tuning teaches the **format** of a helpful answer. Now it produces a full step-by-step solution and commits to a final value.


In [10]:
show(runs, 'Pretrained + SFT')

<IPython.core.display.Latex object>

### Stage 3 — + MPO

Preference optimization tunes **which** answers the model favours. The solution stays structured but gets tighter and more direct.


In [11]:
show(runs, 'Pretrained + SFT + MPO')

<IPython.core.display.Latex object>

### Stage 4 — + GSPO

A final reinforcement-learning stage. The model reasons the most here — exploring more than one route before settling on an answer.


In [12]:
show(runs, 'Pretrained + SFT + MPO + GSPO')

<IPython.core.display.Latex object>

## 4. Budget Forcing

_Budget forcing_ is a blunt but effective lever: rather than letting the model decide when to stop thinking, we cut it short or push it to keep going. The s1 recipe edits the thinking stream directly — append the end-of-thinking token to stop early, or strip it and inject "Wait" to think longer.


### 4.1 Setup

We use a small open reasoning model `DeepSeek-R1-Distill-Qwen-1.5B` so whole demo can run on a free Colab GPU. The trick that makes forcing possible: we generate the thinking stream in short chunks, so we can step in between chunks and edit it — cutting it off, or pushing it to continue.


In [ ]:
# Config
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [ ]:
import torch
from cachesaver.models.transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

# The end-of-thinking delimiter for R1-Distill is the literal "</think>".
THINK_END = "</think>"
# Token id(s) for "</think>" so we can detect/suppress it during decoding.
THINK_END_IDS = tokenizer.encode(THINK_END, add_special_tokens=False)
print("</think> encodes to token ids:", THINK_END_IDS)
print("Loaded", MODEL_ID)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

</think> encodes to token ids: [151649]
Loaded deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B


In [ ]:
def build_prompt(question: str) -> str:
    """R1-Distill chat format. DeepSeek recommends no system prompt and forcing the
    response to begin with <think> so the model actually reasons."""
    messages = [{"role": "user", "content": question +
                 "\nPlease reason step by step, and put your final answer within \\boxed{}."}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return text + "<think>\n"   # force entry into the thinking phase


def budget_forced_generate(question,
                           max_thinking_tokens=None,   # cap (None = unlimited)
                           min_thinking_tokens=0,      # floor (0 = no forced extension)
                           num_waits=0,                # max number of "Wait" injections
                           chunk=64,                   # tokens generated per step
                           max_answer_tokens=256,
                           temperature=0.6):
    """Generate with optional budget forcing. Returns dict with the trace + metadata.

    Mechanism:
      - Generate the <think> span in chunks.
      - If </think> appears but we're under min_thinking_tokens and have waits left,
        strip it and append 'Wait' (FORCE MORE).
      - If we hit max_thinking_tokens first, append '</think>\n\n**Final Answer:**'
        (FORCE LESS), then generate the answer.
    """
    import torch
    text = build_prompt(question)
    thinking_tokens = 0
    waits_used = 0
    ended_naturally = False
    forced_stop = False

    while True:
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=chunk, do_sample=True,
                temperature=temperature, top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                    skip_special_tokens=True)
        text += new_text
        thinking_tokens += chunk

        # Did the model try to end thinking?
        if THINK_END in text:
            # FORCE MORE: under the floor and waits remaining -> suppress + "Wait"
            if thinking_tokens < min_thinking_tokens and waits_used < num_waits:
                text = text.split(THINK_END)[0].rstrip() + "\nWait, "
                waits_used += 1
                continue
            ended_naturally = True
            break

        # FORCE LESS: over the cap -> inject end-of-thinking + answer cue
        if max_thinking_tokens is not None and thinking_tokens >= max_thinking_tokens:
            text = text.rstrip() + "\n" + THINK_END + "\n\n**Final Answer:**\n\n"
            forced_stop = True
            break

        # Safety valve so the loop can't run forever in a demo
        if thinking_tokens >= 8192:
            text = text.rstrip() + "\n" + THINK_END + "\n\n**Final Answer:**\n\n"
            forced_stop = True
            break

    # Generate the answer portion after thinking has ended (naturally or forced)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_answer_tokens, do_sample=True,
            temperature=temperature, top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    full = text + answer

    return {
        "full_text": full,
        "thinking_tokens_generated": thinking_tokens,
        "waits_used": waits_used,
        "ended_naturally": ended_naturally,
        "forced_stop": forced_stop,
        "answer_tail": answer,
    }


### 4.2 Easy problem, unlimited budget

A trivial question (`17 + 25`) with no cap on thinking. Left to its own devices, the model spends a long chain of thought on something it could answer immediately.

Insights:

- With unlimited budget the model _overthinks_: many thinking tokens for a one-step problem.
- More thinking isn't free — it is latency and tokens spent with no gain in accuracy here.


In [ ]:
EASY_Q = "What is 17 + 25?"

r1 = budget_forced_generate(EASY_Q, max_thinking_tokens=None)  # unlimited
print("Thinking tokens generated:", r1["thinking_tokens_generated"])
print("Boxed answer:", extract_boxed(r1["full_text"]))
print(f"---Response---")
print(r1["full_text"])

Thinking tokens generated: 1472
Boxed answer: 42
---Response---
<｜begin▁of▁sentence｜><｜User｜>What is 17 + 25?
Please reason step by step, and put your final answer within \boxed{}.<｜Assistant｜><think>
<think>
Okay, so I need to figure out what 17 plus 25 is. Hmm, let me think about how to approach this. I remember that adding two numbers together involves combining their values. Maybe I can break down the numbers to make the addition easier. 

Let's see, 17 and 25. Both of these numbers are two-digit numbers, which might make it a bit tricky if I try to add them directly. Wait, no, actually, they are both two-digit, but 17 is smaller than 25. Maybe I can start by adding the ones place and then the tens place. 

So, 17 has a 7 in the ones place and a 1 in the tens place. 25 has a 5 in the ones place and a 2 in the tens place. If I add the ones place first, that would be 7 plus 5. Let me do that: 7 plus 5 is 12. Hmm, okay, so I have 12 in the ones place. But wait, 12 is more than 10, so 

### 4.3 Same easy problem, forced budget

Now we _force less_: cap thinking at zero tokens, so the model is pushed straight to its answer.

Insights:

- Cutting the thinking short still lands the correct answer — at a fraction of the tokens.
- For an easy problem, that long chain of thought was wasted budget.


In [ ]:
r2 = budget_forced_generate(EASY_Q, max_thinking_tokens=64)
print("Thinking tokens generated:", r2["thinking_tokens_generated"],
        "(forced_stop =", r2["forced_stop"], ")")
print("Boxed answer:", extract_boxed(r2["full_text"]))
print(f"---Response---")
print(r2["full_text"])

Thinking tokens generated: 64 (forced_stop = True )
Boxed answer: 42
---Response---
<｜begin▁of▁sentence｜><｜User｜>What is 17 + 25?
Please reason step by step, and put your final answer within \boxed{}.<｜Assistant｜><think>
<think>
Alright, so I need to figure out what 17 plus 25 is. Hmm, okay, let me think about how to approach this. I remember when I was learning addition, sometimes it helps to break numbers down into smaller parts to make the addition easier. Maybe I can use that method here.

First
</think>

**Final Answer:**

17 + 25 = 42

Step-by-step explanation:

1. Start with the two numbers you need to add: 17 and 25.
2. Add the units place: 7 + 5 = 12. Write down the 2 and carry over the 1.
3. Add the tens place: 1 + 2 = 3, then add the carried over 1, which makes 4.
4. Combine the results: 42.

So, 17 + 25 equals \boxed{42}.


### 4.4 Hard problem, extended thinking

A genuinely harder problem. We compare a short, unforced attempt against one where we _force more_, stripping the stop token and injecting "Wait" so the model keeps reasoning.

Insights:

- Left short, the model tends to stop early and miss the answer.
- Forcing it to keep thinking (the "Wait" injections) gives it room to recover and reach the correct result.


In [ ]:
HARD_Q = "Let f(x) = x^2 - 6x + 5. For how many integer values of x is f(x) negative?"


# First: short/no extension (likely to stop early)
short = budget_forced_generate(HARD_Q, max_thinking_tokens=256)
# Then: forced extension with Wait injections
extended = budget_forced_generate(HARD_Q, min_thinking_tokens=1500, num_waits=4)

print("SHORT  -> thinking:", short["thinking_tokens_generated"],
        "| answer:", extract_boxed(short["full_text"]))
print("EXTEND -> thinking:", extended["thinking_tokens_generated"],
        "| waits:", extended["waits_used"],
        "| answer:", extract_boxed(extended["full_text"]))
print("\n(Correct answer: f(x)<0 for 1<x<5 -> x in {2,3,4} -> 3 values.)")

SHORT  -> thinking: 256 | answer: 4
EXTEND -> thinking: 1984 | waits: 2 | answer: 3

(Correct answer: f(x)<0 for 1<x<5 -> x in {2,3,4} -> 3 values.)


# 5. Multilinguality in LLM reasoning

Improving reasoning isn't only about quality; it's about **accessibility** too: does that same quality reach everyone, whatever language they ask in?

Does a model actually _reason_ in the language you ask it in? We follow one MGSM problem across languages and model types.A plain chat model, a SOTA reasoning model's real traces, and a live thinking-vs-output view. First we set up the model and a quick English baseline.


In [ ]:
llm = LLM()
print('Model:', llm.model)
print('Languages:', ', '.join(M.LANGUAGES))

In [ ]:
PROBLEM = 5   # which MGSM problem (0–249); the same index is the same problem in every language

prob_en = M.load_mgsm('en', n=PROBLEM + 1)[PROBLEM]
r_en = await M.asolve_direct(llm, prob_en.question, 'en')

print('REQUEST:')
print(r_en.detail['prompt'])
print('\nRESPONSE:')
print(r_en.text)
ok = 'correct' if M.is_correct(r_en.answer, prob_en.answer) else 'wrong'
print(f'\n[extracted: {r_en.answer}  |  gold: {prob_en.answer}  |  {ok}]')

## 5.1 Solving MGSM

MGSM was composed by manually translating the 250 samples of GSM8K in ten different languages. Let's have a look at a couple of them.

Insights:

- The model responds in the language it was asked
- The model's performance degrades.


In [ ]:
langs = ['fr', 'zh']   # French, Russian, Chinese
probs = {lang: M.load_mgsm(lang, n=PROBLEM + 1)[PROBLEM] for lang in langs}

# solve the same problem in each language, concurrently
results = await asyncio.gather(*(M.asolve_direct(llm, probs[l].question, l) for l in langs))

# print the full request (prompt sent) and response (model output) per language
for lang, r in zip(langs, results):
    name = M.LANGUAGES[lang]['name']
    ok = 'correct' if M.is_correct(r.answer, probs[lang].answer) else 'wrong'
    print(f'========== {name} ({lang}) ==========')
    print('REQUEST:')
    print(r.detail['prompt'])
    print('\nRESPONSE:')
    print(r.text)
    print(f'\n[extracted: {r.answer}  |  gold: {probs[lang].answer}  |  {ok}]\n')

### 5.2 Using a SOTA model

In this section we're using a specialized model which in math datasets competes and overcomes state-of-the-art models such as OpenAI's o1 and DeepSeek-R1.

Insights:

- The model no longer responds in the language it was asked
- The model performs quite well in the foreign languages but not as well as in English


In [ ]:
SIZE, BUDGET = '32B', 8000   # which s1.1 model + max thinking budget to load

# load the real s1.1 trace for the same PROBLEM in every language
s1_traces = {lang: T.load_trace(SIZE, lang, BUDGET, doc_id=PROBLEM) for lang in langs}

for lang in langs:
    tr = s1_traces[lang]
    ok = 'correct' if tr.correct else 'WRONG'
    print(f'========== {M.LANGUAGES[lang]["name"]} ({lang}) — s1.1-{SIZE} ==========')
    print('REQUEST:')
    print(M.build_prompt(tr.question, lang))
    print('\nRESPONSE (s1.1 reasoning):')
    print(tr.cot)
    print(f'\n[extracted: {tr.pred}  |  gold: {tr.gold}  |  {ok}]\n')

### 5.3 Inside look

Let's have a look at a particular trace where the model is asked the question in Russian and tries to respond in English.

Insights:

- The chain of thought is in English, but the final answer is then translated into Russian
- The model burns reasoning budget on grammar not math (thinks about Russian, not in it)


In [ ]:
tr = T.load_trace('32B', 'ru', budget=8000, doc_id=5)
print('Russian problem:', tr.question[:90], '...')
print('gold:', tr.gold, ' s1 answer:', tr.pred, ' correct:', tr.correct)
display(HTML(T.highlight_html(tr.cot, title='s1.1-32B reasoning on a Russian MGSM problem')))

### 5.4 Using a reasoning model

Let's investigate a particular case of a reasoning model and specifically its **thinking** and **output** processes as two separate channels.

Insights:

- With the Spanish problem, the model thinks in English and responds in Spanish
- With the Bengali problem, the model does no thinking and directly responds in Bengali
- With the Chinese problem, the model thinks and responds in Chinese


In [ ]:
MODELS = [
    "deepseek/deepseek-v4-flash"
]
COMPARE_LANGS = ['es', 'bn', 'zh']   # languages to compare
MAX_TOKENS = 4096                    # high enough for reasoning models to finish thinking

clients = {m: LLM(model=m) for m in MODELS}
qs = {l: M.load_mgsm(l, n=PROBLEM + 1)[PROBLEM] for l in COMPARE_LANGS}

# run every (language, model) pair concurrently; tolerate per-model failures
async def _run(lang, m):
    try:
        r = await M.asolve_direct(clients[m], qs[lang].question, lang, max_tokens=MAX_TOKENS)
        return lang, m, r, None
    except Exception as e:
        return lang, m, None, f'{type(e).__name__}: {e}'

pairs = await asyncio.gather(*(_run(l, m) for l in COMPARE_LANGS for m in MODELS))
grid = {(l, m): (r, err) for l, m, r, err in pairs}

# grouped first by language, then by model — print THINKING and OUTPUT text + their token counts
for lang in COMPARE_LANGS:
    name = M.LANGUAGES[lang]['name']
    print(f'################### {name} ({lang})   (gold: {qs[lang].answer}) ###################')
    print('REQUEST:')
    print(M.build_prompt(qs[lang].question, lang))
    print()
    for m in MODELS:
        r, err = grid[(lang, m)]
        print(f'----- {m} -----')
        if err:
            print(f'[ERROR: {err}]\n')
            continue
        u = r.usage
        thinking = r.detail.get('reasoning', '')
        output = r.detail.get('content', '') or r.text
        print(f'THINKING:')
        print(thinking if thinking else '[none exposed by this model]')
        print(f'\nOUTPUT:')
        print(output if output else '[empty response]')
        ok = 'correct' if M.is_correct(r.answer, qs[lang].answer) else 'WRONG'
        print(f'\n[extracted: {r.answer}  |  {ok}]\n')
    print()

## 6. Strict separation: making reasoning auditable

Sections 1 to 5 took _in-LLM reasoning_ as the unit of study. For sequential decision-making under uncertainty (such as clinical dialogue) patching has limits. We walk through three failure modes of LLM-only reasoning on one patient, then introduce **MoBayes** (arXiv:2604.20022): an architectural alternative that lets a deterministic Bayesian engine own the belief state and restricts the LLM to language.

All five demos in §6 share the same patient: `ddx_039395`, 89F in DDX plus training set, ground truth pulmonary neoplasm.


### 6.1 Three failure modes

LLMs are bad at reasoning under uncertanity. Because they have some problems on:

1. **Numbers.** Numerical confidences are inaccurate and overconfident (Xiong et al. ICLR 2024), coarse-grained (Feng et al. ICLR 2025), and depart from expected-utility-rational choice (Liu et al. 2024).
2. **Multi-turn.** Performance degrades when information arrives across turns vs. all at once (Laban et al. ICLR 2026; MediQ, Li et al. NeurIPS 2024).
3. **Audit.** No exposed posterior, no information value, no replay.


### 6.2 Failure 1a: probability inconsistency

Same DDxPlus patient (chief complaint only), same prompt, ask for P(pulmonary embolism). Five reps at T=0, ten at T=1.


In [ ]:
"""Demo 1: LLM probability inconsistency on a real DDxPlus case.

Same DDxPlus patient (ddx_039395, 89F, gold = pulmonary neoplasm), chief
complaint only, ask P(pulmonary embolism). T=0 should be reproducible; T=1
shows the sampling spread.
"""
import os, re, json, statistics
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                                # reads ./.env if present
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-5.4-nano"

with open("data/ddxplus/patients.jsonl") as f:
    CASE = next(json.loads(l) for l in f
                if json.loads(l).get("hadm_id") == "ddx_039395")
CHIEF = CASE["Current Visit"]["chiefcomplaint"]
AGE, SEX = CASE["age"], CASE["gender"]

PROMPT = (
    f"Patient: {AGE}-year-old {SEX.lower()}.\n"
    f"Chief complaint: \"{CHIEF}\"\n\n"
    f"What is your estimate of the probability (between 0 and 1) that this "
    f"patient has pulmonary embolism? Answer with just one number, no "
    f"explanation."
)
NUM_RE = re.compile(r"[-+]?\d*\.?\d+")

def ask(temperature):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": PROMPT}],
        temperature=temperature,
    )
    text = r.choices[0].message.content.strip()
    m = NUM_RE.search(text)
    val = float(m.group()) if m else None
    if val is not None and val > 1.0:
        val = val / 100.0
    return text, val

def summarize(label, vals):
    print(f"\n{label}")
    print(f"  values: {[f'{v:.2f}' for v in vals]}")
    if len(vals) >= 2:
        print(f"  mean   = {statistics.mean(vals):.3f}")
        print(f"  stdev  = {statistics.stdev(vals):.3f}")
        print(f"  range  = [{min(vals):.2f}, {max(vals):.2f}]  "
              f"(spread {max(vals)-min(vals):.2f})")

print(f"DDxPlus case: {CASE['hadm_id']}, {AGE}{SEX[0]}, "
      f"ground truth = {CASE['_ddxplus_meta']['diagnosis_name']}")
print(f"Chief complaint: \"{CHIEF}\"")
print(f"Question: P(pulmonary embolism | chief complaint)?  Model: {MODEL}")

print("\n--- temperature = 0 (deterministic, N=5) ---")
det = []
for i in range(5):
    text, val = ask(0.0)
    print(f"  trial {i+1}: raw={text!r:10s} -> p={val}")
    if val is not None: det.append(val)

print("\n--- temperature = 1.0 (sampling, N=10) ---")
samp = []
for i in range(10):
    text, val = ask(1.0)
    print(f"  trial {i+1:2d}: raw={text!r:10s} -> p={val}")
    if val is not None: samp.append(val)

summarize("Deterministic (T=0) summary:", det)
summarize("Sampling (T=1.0) summary:", samp)

print("\nSame model, same prompt, same DDxPlus patient.")
print("The LLM has no stable internal probability; each sample is a new belief.")


KeyError: 'OPENAI_API_KEY'

Insights: T=0 should be reproducible but gives two different values (`0.35` once, `0.25` four times); T=1 spans `0.20` to `0.35`. The LLM's "probability" is a sample from its text distribution, not a stable belief.


### 6.3 Failure 1b: coarse, unanchored rankings

Same patient, this time with the full positive findings. Ask the model to rank the five most likely diagnoses with decimal probabilities. Repeat five times.


In [ ]:
"""Demo 2: coarse / uncalibrated LLM rankings on the same DDxPlus case.

Ask the model to rank top-5 differentials for the patient five times, then
compare against the DDxPlus empirical prior loaded from priors.csv.
"""
import os, re, csv, json
from collections import OrderedDict
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-5.4-nano"
N_TRIALS = 5

with open("data/ddxplus/patients.jsonl") as f:
    CASE = next(json.loads(l) for l in f
                if json.loads(l).get("hadm_id") == "ddx_039395")

positive = [
    "hemoptysis (coughing up blood)",
    "right posterior chest wall pain (knife-like, pleuritic)",
    "significant dyspnea",
    "chronic fatigue",
    "loss of appetite / early satiety",
    "unintentional weight loss",
    "current and former smoker (long history)",
    "family history of lung cancer",
]
findings_str = "\n".join(f"  - {p}" for p in positive)

PROMPT = (
    f"Patient: {CASE['age']}-year-old {CASE['gender'].lower()}.\n"
    f"Findings:\n{findings_str}\n\n"
    "List the 5 most likely diagnoses with probabilities as decimals.\n"
    "Use the exact format, one per line, nothing else:\n"
    "Disease: 0.XX"
)
LINE_RE = re.compile(r"^\s*([A-Za-z][A-Za-z0-9 \-/(),'.]+?)\s*:\s*(0?\.\d+)\s*$")

def normalize(name):
    name = re.sub(r"\s*\(.*?\)\s*", "", name).strip().lower()
    name = re.sub(r"\s*,.*$", "", name)
    aliases = {
        "pulmonary neoplasm": "lung cancer",
        "lung carcinoma": "lung cancer",
        "bronchogenic carcinoma": "lung cancer",
        "lung malignancy": "lung cancer",
        "pulmonary tuberculosis": "tuberculosis",
        "tb": "tuberculosis",
    }
    return aliases.get(name, name)

def ask():
    r = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": PROMPT}],
        temperature=0.7,
    )
    out = []
    for ln in r.choices[0].message.content.strip().splitlines():
        m = LINE_RE.match(ln)
        if m:
            out.append((normalize(m.group(1)), float(m.group(2))))
    return out

trials = [ask() for _ in range(N_TRIALS)]

columns = OrderedDict()
for t in trials:
    for name, _ in t:
        columns.setdefault(name, None)
cols = list(columns.keys())

header = "trial  | " + " | ".join(f"{c[:14]:14s}" for c in cols) + " | SUM"
print(header); print("-" * len(header))
distinct_probs = set()
sums = []
for i, t in enumerate(trials, 1):
    row = {n: p for n, p in t}
    cells = [(f"{row[c]:.2f}".ljust(14) if c in row else "-".ljust(14))
             for c in cols]
    s = sum(row.values())
    sums.append(s)
    for v in row.values(): distinct_probs.add(round(v, 3))
    print(f"T{i:<5d} | " + " | ".join(cells) + f" | {s:.2f}")

print(f"\nDistinct probability values across {N_TRIALS} trials: "
      f"{len(distinct_probs)} -> {sorted(distinct_probs)}")
print(f"Row sums: {[round(s, 2) for s in sums]}  "
      f"({sum(abs(s-1.0)<=0.02 for s in sums)}/{N_TRIALS} within 0.02 of 1.0)")

top1 = [max(t, key=lambda x: x[1])[0] for t in trials]
top1_probs = [max(t, key=lambda x: x[1])[1] for t in trials]
print(f"P(rank-1) across trials: {[round(p,2) for p in top1_probs]}  "
      f"(min={min(top1_probs):.2f}, max={max(top1_probs):.2f})")
print(f"Top-1 disease across trials: {set(top1)} -> "
      f"consistent: {len(set(top1)) == 1}")

ddx_prior = None
with open("data/ddxplus/priors.csv") as f:
    for row in csv.DictReader(f):
        if row["disease_id"] == "d_pulmonary_neoplasm":
            ddx_prior = float(row["prior"]); break

lc_probs = [p for t in trials for n, p in t if n == "lung cancer"]
if lc_probs and ddx_prior:
    llm_mean = sum(lc_probs) / len(lc_probs)
    print(f"\nLung-cancer anchor check:")
    print(f"  DDxPlus empirical prior (d_pulmonary_neoplasm): {ddx_prior:.4f}")
    print(f"  LLM mean P(lung cancer) over {len(lc_probs)} mentions: "
          f"{llm_mean:.4f}")
    print(f"  Ratio LLM / empirical: {llm_mean/ddx_prior:.1f}x")


Insights:

- Only 12 distinct probability values across 5 trials. The LLM picks labels, not numbers.
- 4 of 5 trials produce probabilities that **do not sum to 1** (the LLM is not constrained by basic probability axioms).
- The LLM's mean P(lung cancer) is ~28x the DDxPlus empirical prior (0.014). No anchor in actual epidemiology.


### 6.4 Failure 2: multi-turn degradation

Same patient, same total facts. Two delivery formats: all at once vs. drip-fed through six turns of dialogue. Reproduces the qualitative shape of Laban et al. ICLR 2026 ("LLMs Lost in Multi-Turn Conversation") on one case.


In [ ]:
"""
Demo 3: single-shot vs multi-turn diagnostic degradation.
Case: ddx_039395, 89F, ground truth = Pulmonary neoplasm.
Same model, same total information, different delivery -> different answer.
"""

import os
import re
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-5.4-nano"

# Ground truth patient facts (DDxPlus ddx_039395)
PATIENT = {
    "age": 89,
    "gender": "Female",
    "chief_complaint": (
        "right posterior chest wall pain, coughing up blood, "
        "and a knife-stroke pain"
    ),
    "findings": {
        "hemoptysis (coughing up blood)": "YES",
        "right posterior chest wall pain": "YES",
        "knife-stroke quality pain": "YES",
        "significant dyspnea": "YES",
        "chronic fatigue": "YES",
        "loss of appetite / early satiety": "YES",
        "unintentional weight loss": "YES",
        "current cigarette smoker": "YES",
        "former smoker (long history)": "YES",
        "family history of lung cancer": "YES",
        "recent international travel": "no",
    },
    "ground_truth": "Pulmonary neoplasm",
}


def call(messages, max_tokens=400):
    r = client.chat.completions.create(
        model=MODEL, messages=messages, max_completion_tokens=max_tokens
    )
    return r.choices[0].message.content.strip()


def oracle(question):
    """Answer a clinician question using the feature_map. Specific topics
    come first so 'blood thinners' doesn't get matched as hemoptysis."""
    q = question.lower()
    if any(k in q for k in ["thinner", "anticoag", "warfarin", "apixaban",
                            "rivaroxaban", "aspirin", "clopidogrel", "heparin"]):
        return "no, not on any blood thinners"
    if "travel" in q:
        return "no recent travel"
    if any(k in q for k in ["fever", "chills", "infection", "sick contact"]):
        return "no fever or chills"
    if any(k in q for k in ["smok", "cigar", "tobac"]):
        return "YES, long-time smoker (current and former)"
    if any(k in q for k in ["hemopt", "coughing up blood", "spit blood",
                            "blood in sputum", "blood in cough"]):
        return "YES, coughing up blood"
    if "weight" in q:
        return "YES, unintentional weight loss"
    if any(k in q for k in ["appetite", "eating less", "early satiety"]):
        return "YES, loss of appetite"
    if "famil" in q:
        return "YES, family history of lung cancer"
    if any(k in q for k in ["fatig", "tired", "energy"]):
        return "YES, chronic fatigue"
    if any(k in q for k in ["breath", "dyspn", "short of breath", "winded"]):
        return "YES, significant dyspnea"
    if "chest" in q and "pain" in q:
        return "YES, right posterior chest wall pain, knife-like"
    return "unknown / not documented"


# ---------- (a) SINGLE-SHOT ----------
print("=" * 64)
print("CASE: 89F, ground truth = Pulmonary neoplasm (lung cancer)")
print("=" * 64)

findings_str = "\n".join(f"  - {k}: {v}" for k, v in PATIENT["findings"].items())
single_prompt = f"""Patient: {PATIENT['age']}yo {PATIENT['gender']}.
Chief complaint: {PATIENT['chief_complaint']}.
Documented findings:
{findings_str}

Give the single most likely diagnosis, then one sentence of reasoning.
Format:
DIAGNOSIS: <name>
REASONING: <one sentence>"""

print("\n[A] SINGLE-SHOT (all findings at once)")
print("-" * 64)
single_resp = call([{"role": "user", "content": single_prompt}])
print(single_resp)

# ---------- (b) MULTI-TURN ----------
print("\n[B] MULTI-TURN (LLM gets chief complaint only; must ask)")
print("-" * 64)

system = (
    "You are a physician taking a history from an 89-year-old female patient. "
    "On each turn, EITHER ask ONE focused yes/no clarifying question, "
    "OR if confident, commit to a diagnosis. "
    "Format strictly as one line:\n"
    "  ASK: <question>\n"
    "or\n"
    "  DIAGNOSE: <diagnosis name>"
)
opening_patient = f"I'm an 89-year-old woman. {PATIENT['chief_complaint']}."

history = [
    {"role": "system", "content": system},
    {"role": "user", "content": f"Patient says: \"{opening_patient}\""},
]

asked, final_dx, MAX_TURNS = [], None, 6
for turn in range(1, MAX_TURNS + 1):
    out = call(history, max_tokens=120)
    print(f"\nTurn {turn} doctor: {out}")
    m_dx = re.search(r"DIAGNOSE\s*:\s*(.+)", out, re.IGNORECASE)
    m_q = re.search(r"ASK\s*:\s*(.+)", out, re.IGNORECASE)
    if m_dx:
        final_dx = m_dx.group(1).strip()
        break
    if m_q:
        q = m_q.group(1).strip()
        asked.append(q)
        ans = oracle(q)
        print(f"         patient: {ans}")
        history.append({"role": "assistant", "content": out})
        history.append({"role": "user", "content": f"Patient: {ans}"})
    else:
        history.append({"role": "assistant", "content": out})
        history.append({"role": "user", "content": "Please respond as ASK: or DIAGNOSE:"})

if final_dx is None:
    history.append({"role": "user", "content": "Time's up. Commit now: DIAGNOSE: <name>"})
    forced = call(history, max_tokens=60)
    print(f"\nForced commit: {forced}")
    m = re.search(r"DIAGNOSE\s*:\s*(.+)", forced, re.IGNORECASE)
    final_dx = m.group(1).strip() if m else forced

# ---------- COMPARISON ----------
print("\n" + "=" * 64)
print("COMPARISON")
print("=" * 64)

def hit(text):
    t = text.lower()
    return any(k in t for k in ["pulmonary neoplasm", "lung cancer", "lung ca",
                                "bronchogenic", "lung malignan", "lung tumor"])

joined_q = " | ".join(asked).lower()
covered = {
    "smoking history": any(k in joined_q for k in ["smok", "cigar", "tobac"]),
    "hemoptysis":      any(k in joined_q for k in ["blood", "hemopt"]),
    "weight loss":     "weight" in joined_q,
    "family history":  "famil" in joined_q,
}

print(f"Ground truth   : {PATIENT['ground_truth']}")
print(f"Single-shot dx : {'CORRECT' if hit(single_resp) else 'MISS'}")
print(f"Multi-turn dx  : {final_dx}")
print(f"                 -> {'CORRECT' if hit(final_dx or '') else 'MISS'}")
print(f"Turns used     : {len(asked)} questions before commit")
print("Key oncology features the multi-turn doctor asked about:")
for k, v in covered.items():
    print(f"  {k:18s}: {'YES' if v else 'NO'}")
print(f"  -> {sum(covered.values())}/4 elicited in {len(asked)} turns")

CASE: 89F, ground truth = Pulmonary neoplasm (lung cancer)

[A] SINGLE-SHOT (all findings at once)
----------------------------------------------------------------
DIAGNOSIS: Lung cancer (likely post-obstructive/malignant pleural disease)
REASONING: Elderly current smoker with family history plus hemoptysis, pleuritic "knife-stroke" right posterior chest wall pain, significant dyspnea, and marked weight loss strongly suggests an underlying lung malignancy.

[B] MULTI-TURN (LLM gets chief complaint only; must ask)
----------------------------------------------------------------

Turn 1 doctor: ASK: Have you had any recent fever or chills along with the coughing up blood?
         patient: no fever or chills

Turn 2 doctor: ASK: Is the coughing up blood more than just streaks (e.g., more than a teaspoon or recurring clots)?
         patient: YES, coughing up blood

Turn 3 doctor: ASK: Do you have any shortness of breath or trouble breathing?
         patient: YES, significant dyspnea

Tu

### 6.5 Failure 3: opacity

A fuller LLM doctor on the same case: one question per turn, commits after eight to ten turns. The transcript below is what an audit of this system looks like.


In [ ]:
"""
Demo: A frontier LLM plays doctor on a DDxPlus case.

Shows the baseline "LLM as opaque clinician": chain of thought is hidden,
single final guess at the end, no visible posterior. The MoBayes audit-trace
cell later in this notebook will show the contrast.
"""

import os
import re
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

MODEL = "gpt-5.4-nano"
MAX_TURNS = 10

# DDxPlus case ddx_039395, inlined to avoid a path dependency.
CASE = {
    "age": 89,
    "sex": "female",
    "chief_complaint": (
        "I have right posterior chest wall pain, been coughing up blood, "
        "and a knife stroke pain"
    ),
    "ground_truth": "Pulmonary neoplasm",
}

# Ordered (specific -> generic) keyword -> patient answer. First hit wins.
PATIENT_RULES = [
    ("blood thinner", "No, I'm not on any blood thinners."),
    ("anticoagulant", "No, no anticoagulants."),
    ("aspirin",       "No, I don't take aspirin."),
    ("warfarin",      "No."),
    ("apixaban",      "No."),
    ("international travel", "No, no recent international travel."),
    ("recent travel", "No, no recent travel."),
    ("abroad",        "No, I haven't been abroad."),
    ("immobil",       "No, no recent immobilization."),
    ("surgery",       "No, no recent surgery."),
    ("leg swelling",  "No swelling in my legs."),
    ("leg pain",      "No, my legs are fine."),
    ("fever",         "No fever or chills."),
    ("chills",        "No chills."),
    ("infection",     "No signs of infection."),
    ("family history of lung cancer", "Yes, lung cancer runs in my family."),
    ("family history", "Yes, lung cancer runs in my family."),
    ("family",        "Yes, family history of lung cancer."),
    ("tuberculosis",  "No, no history of TB."),
    ("tb ",           "No, no history of TB."),
    ("smoke",         "Yes, I smoke now and I smoked heavily for decades."),
    ("tobacco",       "Yes, current and long-time former smoker."),
    ("cigarette",     "Yes, I smoke and used to smoke heavily."),
    ("pack year",     "Many pack-years, decades of smoking."),
    ("weight loss",   "Yes, I've lost weight without trying."),
    ("losing weight", "Yes, unintentional weight loss."),
    ("weight",        "Yes, I've lost weight without trying."),
    ("appetite",      "Yes, my appetite is poor and I feel full quickly."),
    ("eating",        "I'm eating less, my appetite is poor."),
    ("fatigue",       "Yes, I feel chronically tired."),
    ("tired",         "Yes, chronic fatigue."),
    ("night sweats",  "No night sweats."),
    ("how much blood", "Streaks to small clots, on and off."),
    ("amount of blood", "Streaks to small clots, on and off."),
    ("how long",      "It's been going on for weeks, getting worse."),
    ("when did",      "It started a few weeks ago and has been getting worse."),
    ("duration",      "A few weeks, gradually worsening."),
    ("hemopt",        "Yes, hemoptysis."),
    ("coughing up blood", "Yes, I cough up blood, streaks and small clots."),
    ("cough up",      "Yes, I cough up blood."),
    ("sputum",        "Blood-tinged sputum."),
    ("short of breath", "Yes, I get very short of breath."),
    ("shortness of breath", "Yes, significant shortness of breath."),
    ("dyspn",         "Yes, significant dyspnea."),
    ("breath",        "Yes, I get short of breath easily."),
    ("wheez",         "No wheezing."),
    ("pleuritic",     "Yes, the pain is worse when I breathe in."),
    ("deep breath",   "Yes, it hurts more on deep breaths."),
    ("chest pain",    "Sharp knife-like pain, right posterior chest wall."),
    ("chest",         "Right posterior chest wall, knife-like pain."),
    ("pain",          "Sharp knife-like pain, right posterior chest wall."),
    ("cough",         "Yes, I have a cough and I'm coughing up blood."),
]


def patient_oracle(question: str) -> str:
    q = question.lower()
    for key, answer in PATIENT_RULES:
        if key in q:
            return answer
    return "I don't have any information about that."


SYSTEM_PROMPT = (
    "You are a clinician taking a focused history from an elderly patient. "
    "Each turn output EXACTLY ONE line, in one of these two formats:\n"
    "  QUESTION: <one short clinical question>\n"
    "  DIAGNOSIS: <single most likely diagnosis>\n"
    "Rules:\n"
    "- One question at a time. No multi-part questions, no lists.\n"
    "- Do not explain your reasoning. Do not narrate. No preamble.\n"
    "- Commit to DIAGNOSIS as soon as you have enough information."
)


def parse(reply):
    m = re.search(r"DIAGNOSIS:\s*(.+)", reply, re.IGNORECASE)
    if m:
        return "diagnosis", m.group(1).strip().rstrip(".")
    m = re.search(r"QUESTION:\s*(.+)", reply, re.IGNORECASE)
    if m:
        return "question", m.group(1).strip()
    return "question", reply.strip().splitlines()[-1]


def is_correct(dx):
    if not dx:
        return False
    d = dx.lower()
    return any(k in d for k in ("neoplasm", "lung cancer", "lung ca",
                                "bronchogenic", "malignan"))


def run():
    intro = (
        f"Patient: {CASE['age']}-year-old {CASE['sex']}. "
        f"Chief complaint: \"{CASE['chief_complaint']}\". "
        "Begin history-taking."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": intro},
    ]
    print(f"Case: {CASE['age']}y {CASE['sex']}, chief complaint:")
    print(f"  \"{CASE['chief_complaint']}\"")
    print(f"Ground truth (hidden from LLM): {CASE['ground_truth']}")
    print("-" * 68)

    total_tokens = 0
    diagnosis = None
    turn = 0
    t0 = time.time()

    for turn in range(1, MAX_TURNS + 1):
        force = turn == MAX_TURNS
        if force:
            messages.append({
                "role": "user",
                "content": "Turn limit reached. Commit now: DIAGNOSIS: <name>.",
            })
        resp = client.chat.completions.create(model=MODEL, messages=messages)
        reply = resp.choices[0].message.content.strip()
        total_tokens += resp.usage.total_tokens
        kind, payload = parse(reply)
        messages.append({"role": "assistant", "content": reply})

        if kind == "diagnosis":
            print(f"Turn {turn}  Doctor commits: DIAGNOSIS: {payload}")
            diagnosis = payload
            break
        if force:
            diagnosis = payload
            print(f"Turn {turn}  Doctor (forced): {payload}")
            break

        answer = patient_oracle(payload)
        print(f"Turn {turn}")
        print(f"  Doctor : {payload}")
        print(f"  Patient: {answer}")
        messages.append({"role": "user", "content": answer})

    elapsed = time.time() - t0
    print("-" * 68)
    print(
        f"Turns: {turn} | Tokens: {total_tokens} | Time: {elapsed:.1f}s\n"
        f"Final diagnosis: {diagnosis}\n"
        f"Ground truth   : {CASE['ground_truth']}\n"
        f"Correct        : {is_correct(diagnosis)}\n"
        "Visible posterior over differentials: (none, LLM committed to a single label)"
    )


run()


Case: 89y female, chief complaint:
  "I have right posterior chest wall pain, been coughing up blood, and a knife stroke pain"
Ground truth (hidden from LLM): Pulmonary neoplasm
--------------------------------------------------------------------
Turn 1
  Doctor : When did the coughing up blood start?
  Patient: It started a few weeks ago and has been getting worse.
Turn 2
  Doctor : Do you have shortness of breath or trouble breathing?
  Patient: Yes, significant shortness of breath.
Turn 3
  Doctor : Have you had any fever or chills?
  Patient: No fever or chills.
Turn 4
  Doctor : Is the chest pain worse when you take a deep breath or cough?
  Patient: Yes, I get short of breath easily.
Turn 5
  Doctor : Do you smoke or have a history of smoking?
  Patient: Yes, I smoke now and I smoked heavily for decades.
Turn 6
  Doctor : Have you lost weight without trying?
  Patient: Yes, I've lost weight without trying.
Turn 7
  Doctor : Do you have any history of tuberculosis or prior lung pr

Insights: correct diagnosis, opaque path. No probability over the 48 other DDxPlus candidates, no information value per question, no replay determinism.


### 6.6 The architectural alternative: MoBayes

Three failures, one cause: reasoning inside the LLM. **MoBayes** restricts the LLM to language (parse utterance into `(feature, value, confidence)`, verbalise the engine's chosen question) and hands the belief state, posterior updates, EIG-driven question selection, and τ-thresholded commit/abstain to a deterministic Bayesian engine.

![MoBayes architecture](assets/mobayes_architecture.png)

Every turn is one closed-form numpy update on a posterior vector. The LLM never sees the posterior.


### 6.7 The knowledge base: DDxPlus train split as Dirichlet counts

Likelihoods `P(feature = v | disease)` and prior `P(disease)` are built from DDxPlus directly: 1.03M synthetic records, 49 diseases, 314 features. Co-occurrence counts plus Laplace smoothing give the Dirichlet-Categorical posterior used at inference time. Two consequences:

- **Population-swappable.** Adapting to a new clinic = rebuild the KB. The LLM is not retrained.


### 6.8 The MoBayes audit trace

Same patient as §6.5. Same outcome. The contrast is in _what the cells show you_. Two cells: a **setup** that extracts the KB from the DDxPlus train split, loads the patient, and prints the three deployment knobs (τ, min/max questions), and an **EIG loop** that runs the asking pass until the stopping rule fires. Every printed line is one closed-form numpy update; re-running both cells produces the same trace byte for byte.


In [ ]:
# MoBayes setup: extract the KB from DDxPlus train split, load the patient,
# expose the three deployment knobs the operator actually controls.
import json
import numpy as np
from mobayes import load_kb, StoppingRules, resolve_confidence, eig_over

# --- 1) KB extraction (Dirichlet-Categorical from DDxPlus train counts) ---
wm = load_kb('data/ddxplus')
leaves = wm.leaves
name_of = {fid: s['name'] for fid, s in wm.schema.items()}

print(f'Knowledge base (built from DDxPlus train split):')
print(f'  diseases  : {wm.num_diseases}')
print(f'  features  : {len(name_of)}')
print(f'  P(feature|disease) from Dirichlet posterior (count + 1) / (sum + |values|)')
print(f'  P(disease) empirical, from training-split frequencies')

# --- 2) Patient (held-out test profile) ---
with open('data/ddxplus/patients.jsonl') as f:
    CASE = next(json.loads(l) for l in f
                if json.loads(l).get('hadm_id') == 'ddx_039395')
GOLD = CASE['_ddxplus_meta']['ground_truth_ids'][0]
GOLD_IDX = leaves.index(GOLD)
FMAP = CASE['_ddxplus_meta']['feature_map']

def parse_finding(raw):
    if raw.endswith(': YES'): return ('yes', 'very_likely')
    if raw.endswith(': no'):  return ('no',  'very_likely')
    return None
PATIENT = {fid: parse_finding(raw) for fid, raw in FMAP.items()
           if parse_finding(raw) is not None}

print(f'\nPatient: {CASE["hadm_id"]} ({CASE["age"]}{CASE["gender"][0]})')
print(f'  ground truth: {GOLD}  (empirical prior rank: '
      f'{1 + int(np.sum(wm.posterior() > wm.posterior()[GOLD_IDX]))}/{wm.num_diseases})')
print(f'  positive findings in profile: {len(PATIENT)}')

# --- 3) Engine parameters (the deployment knobs) ---
TAU = 0.85            # commit threshold; slide for accuracy / coverage trade-off
MIN_QUESTIONS = 0     # warm-up: minimum turns before commit allowed
MAX_QUESTIONS = 8     # hard budget ceiling
stop = StoppingRules(confidence_threshold=TAU,
                     max_questions=MAX_QUESTIONS,
                     min_questions_before_abstain=MIN_QUESTIONS)

print(f'\nMoBayes parameters:')
print(f'  tau (commit threshold)        = {TAU}')
print(f'  min questions before commit   = {MIN_QUESTIONS}')
print(f'  max questions budget          = {MAX_QUESTIONS}')
print(f'  -> commits when max posterior >= {TAU} OR question budget exhausted')


In [ ]:
# MoBayes loop: opening intake (chief-complaint -> 2 features), then EIG-driven
# questions until the stopping rule fires. Every printed line is one closed-form
# numpy update on the posterior vector.

def bars(p, k=5):
    idx = np.argsort(p)[::-1][:k]
    return '\n'.join(
        f"  {p[i]:.3f}  {'#' * int(p[i] * 60):<24s}  {leaves[i].replace('d_','')}"
        for i in idx)

# --- PRIOR ---
print('=' * 78)
print(f'PRIOR (empirical, {wm.num_diseases} diseases)   H = {wm.entropy():.3f} bits')
print('=' * 78)
print(bars(wm.posterior()))

# --- INTAKE (chief complaint parsed into 2 features) ---
PRIOR_H = wm.entropy()
INTAKE = [('f_e_45', 'yes', 'very_likely'),   # been coughing up blood
          ('f_e_66', 'yes', 'very_likely')]   # significant dyspnea
for fid, v, lab in INTAKE:
    wm.update(fid, v, confidence=resolve_confidence(lab))
print()
print('=' * 78)
print(f'INTAKE: hemoptysis + dyspnea parsed   H = {wm.entropy():.3f}  '
      f'(dropped {PRIOR_H - wm.entropy():.3f} bits)')
print('=' * 78)
print(bars(wm.posterior()))

# --- EIG LOOP, restricted to features the patient profile records ---
for turn in range(1, MAX_QUESTIONS + 1):
    candidates = [f for f in PATIENT if f not in wm.asked]
    if not candidates: break
    fid, eig = eig_over(wm, candidates)
    val, lab = PATIENT[fid]
    H_before = wm.entropy()
    wm.update(fid, val, confidence=resolve_confidence(lab))
    print()
    print('=' * 78)
    print(f'TURN {turn}   EIG = {eig:.3f} bits')
    print('=' * 78)
    print(f'  Q: {name_of[fid]}')
    print(f'  A: {val}  (parsed confidence: {lab})')
    print(f'  H: {H_before:.3f} -> {wm.entropy():.3f}  '
          f'({wm.entropy()-H_before:+.3f} bits)')
    print(bars(wm.posterior()))
    done, why = stop.should_stop(wm, turn)
    if done:
        d = leaves[int(np.argmax(wm.posterior()))]
        ok = 'CORRECT' if d == GOLD else 'WRONG'
        print()
        print(f'>>> COMMIT: {d}  '
              f'(p = {float(np.max(wm.posterior())):.3f}, reason = {why})')
        print(f'>>> Ground truth: {GOLD}  ->  {ok}')
        break


### 6.9 What did separation buy

Same patient, both pipelines committed correctly. Only one tells you how. Three further properties (Figure 3 of the paper):

![Cost vs accuracy](assets/mobayes_cost.png)

**Cost.** Small sensor LLM + engine beats much larger standalone doctors at ~10x lower per-token cost.

![Controllable selective diagnosis](assets/mobayes_tau.png)

**Controllability.** Sliding τ traces a continuous accuracy / coverage frontier. Standalone LLM is one point; MoBayes is the whole curve.

![Persona robustness](assets/mobayes_persona.png)

**Robustness.** Standalone LLMs collapse under distrustful / dazed / overanxious patient personas; the Bayesian backbone is stable.

And from §5.3 of the paper: the same frontier LLM elicited _once_ into a KB and reasoned over by the engine beats that same LLM used turn by turn as the doctor. The gain is **architectural, not informational**.


Insights: ground truth starts at `p=0.014` (rank 38/49) under the empirical prior. Two intake features + three EIG turns commit at `p>0.97`. The questions the engine picks (chest pain, smoking, weight loss) are exactly the discriminators the multi-turn LLM in §6.4 failed to ask. Every line is replayable.


## Sources

- s1: Simple Test-Time Scaling — arXiv:2501.19393 (budget forcing, "Wait" ablation)
- ThinkPrune — arXiv:2504.01296 (R1-Distill budget-forcing implementation details)
- Follow the Path — arXiv:2505.11140 (budget sweep; ~2048-token plateau)
- Certainty-Guided Reasoning — arXiv:2509.07820 (forcing can degrade on larger models)
- DeepSeek-R1 Thoughtology — arXiv:2504.07128 (thinking-budget adherence)
- vLLM reasoning_budget PR — github.com/vllm-project/vllm/pull/37112
- MoBayes: arXiv:2604.20022 (modular Bayesian framework, strict separation of language and reasoning; case study in §6)
- BED-LLM: arXiv:2508.21184 (Choudhury et al. ICLR 2026, sequential Bayesian experimental design over LLMs)
- LLMs Lost in Multi-Turn Conversation: Laban et al. ICLR 2026
- MediQ: arXiv:2406.00922 (Li et al. NeurIPS 2024, question-asking LLMs for reliable interactive clinical reasoning)
- LLM uncertainty calibration: arXiv:2306.13063 (Xiong et al. ICLR 2024)
- BIRD: trustworthy Bayesian inference for LLMs (Feng et al. ICLR 2025)
- DDxPlus: NeurIPS 2022 Datasets and Benchmarks (Tchango et al.)
